<a href="https://colab.research.google.com/github/SarahkhIT/AgenticAIProject/blob/main/AgenticAIProject/notebooks/04_functional_api_reliability_and_observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Event Planner — 04. Functional API & Reliability + LangSmith Observability

**Program:** Building Agentic AI Systems by SDAIA Academy

**Session Dates:** 9th of August, 2026 - 13th of August, 2026

**Declared Track:** Track A

**Covers:** Rubric 6 — LangGraph Functional API & Error Handling · Rubric 8 — LangSmith Observability

Self-contained: re-declares the core planning tools from Rubric 1, since the `@task`-wrapped functions call them directly. LangSmith tracing is configured before the demo runs so the trace-inspection cell has real data to query.

## Team Members
- Setah Mohammed Alajmi
- Raneem Abdullah Alsheddi
- Jana Hamad Alhumaizi
- Shatha Hamad Bin Mana
- Sarah Abdulaziz Alkhudhiri


In [ ]:
!pip install -q \
    "langchain>=1.0" \
    "langgraph>=1.0" \
    langchain-groq \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    pypdf \
    reportlab

In [ ]:
# ============================================================
# CAPSTONE SECRET SETUP — NO HARDCODED API KEYS
# ============================================================

import os
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

if not groq_key:
    raise RuntimeError(
        "GROQ_API_KEY is missing. Add it to Colab Secrets and enable Notebook access."
    )

os.environ["GROQ_API_KEY"] = groq_key

print("PASS: GROQ_API_KEY loaded securely from Colab Secrets.")


PASS: GROQ_API_KEY loaded securely from Colab Secrets.


### Shared tool definitions (from Rubric 1)
Re-declared here so this notebook runs standalone; see `01_agent_fundamentals.ipynb` for the original section.

In [ ]:
# ============================================================
# SMART EVENT PLANNER
# Person 1 — Agent Fundamentals & Tools
# ============================================================
#
# Capstone requirements addressed in this file:
#
# 1. Agent Fundamentals
#    - Real LLM-based agent
#    - Real tool calls
#    - Tools use their arguments to perform actual work
#    - Structured output with Pydantic
#    - with_structured_output()
#
# 2. Integration-ready
#    - EventRequest is a clear contract
#    - build_event_planning_agent() can be used by Person 2
#    - Tools are independent and reusable
#
# IMPORTANT:
# - Set GROQ_API_KEY as an environment variable.
# - Never put the API key directly in this file.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
from typing import Literal

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# ============================================================
# 1. CONFIGURATION
# ============================================================

DEFAULT_MODEL = "groq:llama-3.3-70b-versatile"


def get_model(model_name: str = DEFAULT_MODEL):
    """
    Create the chat model.

    The API key is read from the environment.
    """

    if not os.environ.get("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY is not set.\n"
            "Please set it as an environment variable before running the agent."
        )

    return init_chat_model(model_name)


# ============================================================
# 2. PYDANTIC MODELS
# ============================================================

class EventRequest(BaseModel):
    """
    Structured representation of the user's event request.

    The LLM populates this model using with_structured_output().
    """

    event_type: str = Field(
        description="Type of event, for example graduation, wedding, birthday."
    )

    guest_count: int = Field(
        gt=0,
        description="Expected number of guests."
    )

    budget: float = Field(
        gt=0,
        description="Total event budget in Saudi Riyals."
    )

    date: str | None = Field(
        default=None,
        description="Event date if explicitly provided by the user."
    )

    location_pref: Literal["indoor", "outdoor"] | None = Field(
        default=None,
        description="Indoor or outdoor preference if explicitly provided."
    )

    theme: str | None = Field(
        default=None,
        description="Event theme or style if explicitly provided."
    )


class BudgetBreakdown(BaseModel):
    venue: float
    catering: float
    decoration: float
    entertainment: float
    logistics: float
    contingency: float


class VenueOption(BaseModel):
    name: str
    capacity: int
    indoor: bool
    estimated_cost: float
    location: str
    reason: str


class CateringOption(BaseModel):
    name: str
    price_per_guest: float
    estimated_total: float
    cuisine: str
    reason: str


class DecorationPlan(BaseModel):
    theme: str
    estimated_cost: float
    concept: str
    items: list[str]


class ChecklistItem(BaseModel):
    task: str
    category: str
    due_before_event_days: int
    priority: Literal["high", "medium", "low"]


# ============================================================
# 3. TOOL 1 — BUDGET CALCULATOR
# ============================================================

@tool
def calculate_budget_split(
    total_budget: float,
    event_type: str,
) -> dict:
    """
    Calculate a realistic event budget allocation.

    Arguments:
        total_budget: Total event budget in SAR.
        event_type: Type of event.

    Returns:
        A structured budget breakdown.
    """

    if total_budget <= 0:
        raise ValueError("total_budget must be greater than zero.")

    if not event_type.strip():
        raise ValueError("event_type cannot be empty.")

    # Default allocation for the Smart Event Planner.
    #
    # The event_type is intentionally accepted as an argument
    # because the LLM must provide it when calling the tool.
    #
    # This can later be customized for different event types.

    allocation = {
        "venue": 0.30,
        "catering": 0.35,
        "decoration": 0.15,
        "entertainment": 0.08,
        "logistics": 0.07,
        "contingency": 0.05,
    }

    breakdown = BudgetBreakdown(
        venue=round(total_budget * allocation["venue"], 2),
        catering=round(total_budget * allocation["catering"], 2),
        decoration=round(total_budget * allocation["decoration"], 2),
        entertainment=round(total_budget * allocation["entertainment"], 2),
        logistics=round(total_budget * allocation["logistics"], 2),
        contingency=round(total_budget * allocation["contingency"], 2),
    )

    return breakdown.model_dump()


# ============================================================
# 4. TOOL 2 — VENUE SEARCH
# ============================================================

@tool
def search_venues(
    guest_count: int,
    budget: float,
    indoor: bool,
) -> list[dict]:
    """
    Find venues based on guest capacity, budget and indoor/outdoor preference.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum venue budget in SAR.
        indoor: True for indoor venues, False for outdoor venues.

    Returns:
        Matching venue options.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    # Demo dataset.
    #
    # These are NOT claimed to be real external venues.
    # They are project data used to demonstrate real filtering
    # and tool execution.
    venues = [
        {
            "name": "Elegant Hall Riyadh",
            "capacity": 120,
            "indoor": True,
            "estimated_cost": 4200,
            "location": "Riyadh",
            "reason": "Elegant indoor hall suitable for formal celebrations.",
        },
        {
            "name": "Garden Celebration Venue",
            "capacity": 150,
            "indoor": False,
            "estimated_cost": 3500,
            "location": "Riyadh",
            "reason": "Outdoor garden venue suitable for large celebrations.",
        },
        {
            "name": "Modern Event Studio",
            "capacity": 90,
            "indoor": True,
            "estimated_cost": 3900,
            "location": "Riyadh",
            "reason": "Modern indoor venue suitable for smaller elegant events.",
        },
        {
            "name": "Grand Celebration Center",
            "capacity": 250,
            "indoor": True,
            "estimated_cost": 5500,
            "location": "Riyadh",
            "reason": "Large indoor event center for bigger celebrations.",
        },
    ]

    # REAL filtering using the tool arguments.
    matching_venues = [
        venue
        for venue in venues
        if (
            venue["capacity"] >= guest_count
            and venue["estimated_cost"] <= budget
            and venue["indoor"] == indoor
        )
    ]

    # Cheapest suitable options first.
    matching_venues.sort(
        key=lambda venue: (
            venue["estimated_cost"],
            venue["capacity"],
        )
    )

    return [
        VenueOption(**venue).model_dump()
        for venue in matching_venues[:3]
    ]


# ============================================================
# 5. TOOL 3 — CATERING SEARCH
# ============================================================

@tool
def search_catering(
    guest_count: int,
    budget: float,
) -> list[dict]:
    """
    Find catering options based on guest count and catering budget.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum catering budget in SAR.

    Returns:
        Catering options that fit the budget.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    catering_options = [
        {
            "name": "Premium Saudi Buffet",
            "price_per_guest": 65,
            "cuisine": "Saudi / Arabic",
            "reason": "Suitable for formal family celebrations.",
        },
        {
            "name": "International Buffet",
            "price_per_guest": 70,
            "cuisine": "International",
            "reason": "Broad menu suitable for mixed preferences.",
        },
        {
            "name": "Elegant Finger Food",
            "price_per_guest": 45,
            "cuisine": "International",
            "reason": "Suitable for a modern elegant reception.",
        },
    ]

    results = []

    for option in catering_options:

        # REAL calculation based on guest_count.
        estimated_total = (
            guest_count * option["price_per_guest"]
        )

        # REAL budget filtering.
        if estimated_total <= budget:

            results.append(
                CateringOption(
                    name=option["name"],
                    price_per_guest=option["price_per_guest"],
                    estimated_total=estimated_total,
                    cuisine=option["cuisine"],
                    reason=option["reason"],
                ).model_dump()
            )

    results.sort(
        key=lambda item: item["estimated_total"]
    )

    return results[:3]


# ============================================================
# 6. TOOL 4 — DECORATION PLANNER
# ============================================================

@tool
def suggest_decoration(
    theme: str,
    guest_count: int,
    budget: float,
) -> dict:
    """
    Suggest a decoration concept based on theme, guest count and budget.

    Arguments:
        theme: Requested event theme.
        guest_count: Number of guests.
        budget: Decoration budget in SAR.

    Returns:
        A structured decoration plan.
    """

    if not theme.strip():
        raise ValueError("theme cannot be empty.")

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    normalized_theme = theme.lower().strip()

    if "elegant" in normalized_theme:

        concept = (
            "Elegant graduation setup with neutral colors, "
            "warm lighting, floral accents and a decorated stage."
        )

        items = [
            "Graduation backdrop",
            "Warm ambient lighting",
            "Floral centerpieces",
            "Welcome signage",
            "Decorated stage",
        ]

    elif "modern" in normalized_theme:

        concept = (
            "Modern minimalist setup with clean lines, "
            "accent lighting and a contemporary photo area."
        )

        items = [
            "Minimalist backdrop",
            "Accent lighting",
            "Modern table styling",
            "Photo wall",
        ]

    elif "casual" in normalized_theme:

        concept = (
            "Casual and welcoming setup with simple colors, "
            "comfortable seating and themed decorations."
        )

        items = [
            "Themed backdrop",
            "Simple table decoration",
            "Welcome signage",
            "Photo area",
        ]

    else:

        concept = (
            f"A {theme} themed decoration concept "
            "adapted to the event size and available budget."
        )

        items = [
            "Themed backdrop",
            "Table decoration",
            "Welcome signage",
            "Photo area",
        ]

    return DecorationPlan(
        theme=theme,
        estimated_cost=round(budget, 2),
        concept=concept,
        items=items,
    ).model_dump()


# ============================================================
# 7. TOOL 5 — EVENT CHECKLIST
# ============================================================

@tool
def create_checklist(
    event_date: str,
    categories: list[str],
) -> list[dict]:
    """
    Create an event preparation checklist.

    Arguments:
        event_date: Event date.
        categories: Planning categories to include.

    Returns:
        Checklist items.
    """

    if not event_date.strip():
        raise ValueError("event_date is required.")

    if not categories:
        raise ValueError("At least one category is required.")

    category_tasks = {

        "venue": [
            (
                "Confirm venue booking",
                30,
                "high",
            ),
            (
                "Confirm seating layout",
                7,
                "medium",
            ),
        ],

        "catering": [
            (
                "Confirm catering menu",
                14,
                "high",
            ),
            (
                "Confirm final guest count",
                3,
                "high",
            ),
        ],

        "decoration": [
            (
                "Finalize decoration concept",
                21,
                "medium",
            ),
            (
                "Confirm decoration setup",
                7,
                "high",
            ),
        ],

        "logistics": [
            (
                "Prepare event timeline",
                7,
                "high",
            ),
            (
                "Confirm equipment and sound system",
                5,
                "medium",
            ),
        ],

        "approval": [
            (
                "Review final event plan with user",
                1,
                "high",
            ),
        ],
    }

    checklist = []

    for category in categories:

        tasks = category_tasks.get(
            category.lower().strip(),
            [],
        )

        for task, days, priority in tasks:

            checklist.append(
                ChecklistItem(
                    task=task,
                    category=category,
                    due_before_event_days=days,
                    priority=priority,
                ).model_dump()
            )

    return checklist


# ============================================================
# 8. REGISTER ALL TOOLS
# ============================================================

ALL_TOOLS = [
    calculate_budget_split,
    search_venues,
    search_catering,
    suggest_decoration,
    create_checklist,
]


# ============================================================
# 9. STRUCTURED INTAKE
# ============================================================

INTAKE_SYSTEM_PROMPT = """
You are the structured intake component of Smart Event Planner.

Extract event-planning information from the user's message.

IMPORTANT:
Only extract information that is explicitly supported by the user's
message.

DO NOT invent or assume:
- event dates
- guest counts
- budgets
- indoor/outdoor preferences
- themes

The required event fields are:
- event_type
- guest_count
- budget

The optional fields are:
- date
- location_pref
- theme

If an optional field is not mentioned, return null.

The budget is represented in Saudi Riyals.
"""


def extract_event_request(
    user_message: str,
    model=None,
) -> EventRequest:
    """
    Convert natural-language user input into a validated EventRequest.

    This uses LangChain structured output with a Pydantic model.
    """

    if not user_message.strip():
        raise ValueError(
            "user_message cannot be empty."
        )

    model = model or get_model()

    structured_model = model.with_structured_output(
        EventRequest
    )

    event = structured_model.invoke(
        [
            {
                "role": "system",
                "content": INTAKE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ]
    )

    return event


# ============================================================
# 10. CHECK WHETHER EVENT DETAILS ARE COMPLETE
# ============================================================

def get_missing_event_fields(
    event: EventRequest,
) -> list[str]:
    """
    Return optional fields that are useful for complete planning.

    This does NOT invent missing information.
    """

    missing = []

    if event.date is None:
        missing.append("date")

    if event.location_pref is None:
        missing.append("location_pref")

    if event.theme is None:
        missing.append("theme")

    return missing


def build_clarifying_question(
    event: EventRequest,
) -> str | None:
    """
    Build a follow-up question when optional planning details
    are missing.
    """

    missing = get_missing_event_fields(event)

    if not missing:
        return None

    questions = []

    if "date" in missing:
        questions.append(
            "What date is the event?"
        )

    if "location_pref" in missing:
        questions.append(
            "Would you prefer indoor or outdoor?"
        )

    if "theme" in missing:
        questions.append(
            "What style or theme would you like?"
        )

    return (
        "Before I build the full plan, "
        "please provide: "
        + " ".join(questions)
    )


# ============================================================
# 11. AGENT SYSTEM PROMPT
# ============================================================

AGENT_SYSTEM_PROMPT = """
You are the core event-planning agent for Smart Event Planner.

Your job is to create a practical event plan using the available tools.

IMPORTANT RULE:
Use the tools to obtain planning information.
Do NOT invent:
- budget calculations
- venue options
- catering prices
- decoration costs
- checklist items

The available tools are:

1. calculate_budget_split
   Use it to calculate the event budget allocation.

2. search_venues
   Use it to find suitable venues based on guest count,
   venue budget and indoor/outdoor preference.

3. search_catering
   Use it to find catering options based on guest count
   and catering budget.

4. suggest_decoration
   Use it to create a decoration concept based on theme,
   guest count and decoration budget.

5. create_checklist
   Use it to create event preparation tasks.

You should call the tools yourself when needed.

The tool results are the source of truth for numerical
recommendations.

When all relevant information is available, produce a concise
event plan containing:

1. Event summary
2. Budget breakdown
3. Recommended venue
4. Catering recommendation
5. Decoration concept
6. Preparation checklist

Do not claim that the venue or catering data came from a real
external service. The current tools use the project's local
planning dataset.

If a required detail is missing, clearly state that the information
is needed rather than inventing it.
"""


# ============================================================
# 12. BUILD THE REAL AGENT
# ============================================================

def build_event_planning_agent(
    model=None,
):
    """
    Build the reusable Smart Event Planner agent.

    Person 2 can later integrate this agent into the project's
    multi-agent routing architecture.
    """

    model = model or get_model()

    agent = create_agent(
        model=model,
        tools=ALL_TOOLS,
        system_prompt=AGENT_SYSTEM_PROMPT,
    )

    return agent


# ============================================================
# 13. RUN THE AGENT
# ============================================================

def run_event_planner(
    event: EventRequest,
    model=None,
):
    """
    Run the event planning agent using a validated EventRequest.
    """

    # The planning agent needs these details.
    if event.date is None:
        raise ValueError(
            "event.date is required before running the full planner."
        )

    if event.location_pref is None:
        raise ValueError(
            "event.location_pref is required before running the full planner."
        )

    if event.theme is None:
        raise ValueError(
            "event.theme is required before running the full planner."
        )

    agent = build_event_planning_agent(model)

    event_details = event.model_dump()

    planning_prompt = f"""
Create the event plan using the following structured event request:

{event_details}

You must use the available tools to calculate and retrieve
the planning information.

Do not invent tool results.

Use the actual tool outputs in your final response.
"""

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": planning_prompt,
                }
            ]
        }
    )

    return result


# ============================================================
# 14. DISPLAY REAL TOOL CALLS
# ============================================================

def display_agent_trace(
    result,
):
    """
    Display the actual LLM -> Tool -> Tool Result execution.

    This is important evidence for the Capstone rubric because
    it demonstrates that the LLM selected and called the tools.
    """

    print("\n")
    print("=" * 80)
    print("REAL AGENT TOOL-CALL TRACE")
    print("=" * 80)

    tool_call_count = 0

    for message in result["messages"]:

        # ----------------------------------------------------
        # AI MESSAGE
        # ----------------------------------------------------

        if message.type == "ai":

            tool_calls = getattr(
                message,
                "tool_calls",
                [],
            )

            if tool_calls:

                for call in tool_calls:

                    tool_call_count += 1

                    print("\n[LLM -> TOOL]")
                    print(
                        f"Tool: {call['name']}"
                    )

                    print(
                        "Arguments:"
                    )

                    print(
                        call["args"]
                    )

            elif message.content:

                print("\n[LLM FINAL RESPONSE]")
                print(message.content)

        # ----------------------------------------------------
        # TOOL MESSAGE
        # ----------------------------------------------------

        elif message.type == "tool":

            print("\n[TOOL -> LLM]")

            print(
                f"Tool: {message.name}"
            )

            print(
                "Result:"
            )

            print(
                message.content
            )

    print("\n")
    print("=" * 80)
    print(
        f"TOTAL TOOL CALLS: {tool_call_count}"
    )
    print("=" * 80)

    return tool_call_count


# ============================================================
# 15. OFFLINE TOOL TEST
# ============================================================
#
# This section verifies that the tools themselves work.
#
# IMPORTANT:
# These are MANUAL tool calls.
# They are NOT the evidence for LLM tool selection.
#
# The actual Agent evidence is the test below this section.
# ============================================================

def run_offline_tool_tests():

    print("\n")
    print("=" * 80)
    print("OFFLINE TOOL TESTS")
    print("=" * 80)

    budget = calculate_budget_split.invoke(
        {
            "total_budget": 15000,
            "event_type": "graduation",
        }
    )

    print("\nBudget:")
    print(budget)

    venues = search_venues.invoke(
        {
            "guest_count": 80,
            "budget": budget["venue"],
            "indoor": True,
        }
    )

    print("\nVenues:")
    for venue in venues:
        print(venue)

    catering = search_catering.invoke(
        {
            "guest_count": 80,
            "budget": budget["catering"],
        }
    )

    print("\nCatering:")
    for option in catering:
        print(option)

    decoration = suggest_decoration.invoke(
        {
            "theme": "elegant",
            "guest_count": 80,
            "budget": budget["decoration"],
        }
    )

    print("\nDecoration:")
    print(decoration)

    checklist = create_checklist.invoke(
        {
            "event_date": "2026-09-20",
            "categories": [
                "venue",
                "catering",
                "decoration",
                "logistics",
                "approval",
            ],
        }
    )

    print("\nChecklist:")
    for item in checklist:
        print(item)

    print("\nOffline tool tests completed successfully.")


# ============================================================
# 16. CAPSTONE DEMO
# ============================================================

def run_capstone_demo():

    print("\n")
    print("#" * 80)
    print("# SMART EVENT PLANNER — PERSON 1 CAPSTONE DEMO")
    print("#" * 80)

    # --------------------------------------------------------
    # USER INPUT
    # --------------------------------------------------------

    user_message = """
    I want to plan a graduation party for 80 guests
    with a budget of 15,000 SAR.
    The event will be on 20 September 2026.
    I prefer an indoor venue with an elegant theme.
    """

    print("\n")
    print("=" * 80)
    print("USER REQUEST")
    print("=" * 80)

    print(user_message)

    # --------------------------------------------------------
    # STRUCTURED INTAKE
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("STRUCTURED OUTPUT — EVENT REQUEST")
    print("=" * 80)

    event = extract_event_request(
        user_message
    )

    print(
        event.model_dump_json(
            indent=2
        )
    )

    # --------------------------------------------------------
    # CHECK MISSING INFORMATION
    # --------------------------------------------------------

    missing = get_missing_event_fields(
        event
    )

    if missing:

        print("\n")
        print("=" * 80)
        print("MISSING OPTIONAL INFORMATION")
        print("=" * 80)

        print(
            build_clarifying_question(event)
        )

        print(
            "\nThe demo will stop here because the planner "
            "does not invent missing information."
        )

        return

    # --------------------------------------------------------
    # REAL AGENT EXECUTION
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("RUNNING REAL LLM AGENT")
    print("=" * 80)

    result = run_event_planner(
        event
    )

    # --------------------------------------------------------
    # DISPLAY REAL TOOL CALLS
    # --------------------------------------------------------

    tool_call_count = display_agent_trace(
        result
    )

    # --------------------------------------------------------
    # CAPSTONE ASSERTION
    # --------------------------------------------------------

    if tool_call_count == 0:

        raise RuntimeError(
            "The agent completed without calling any tool. "
            "This would NOT satisfy the real tool-calling requirement."
        )

    print("\n")
    print("=" * 80)
    print("CAPSTONE CHECK")
    print("=" * 80)

    print(
        "PASS: The LLM made real tool calls."
    )

    print(
        f"PASS: Number of tool calls = {tool_call_count}"
    )

    print(
        "PASS: Structured EventRequest was produced with Pydantic."
    )

    print(
        "PASS: with_structured_output() was used."
    )

    print(
        "PASS: Tools received arguments from the agent."
    )

    print(
        "PASS: Tool results were returned to the LLM."
    )

    print(
        "PASS: Final plan was generated from tool results."
    )


# ============================================================
# 17. MAIN
# ============================================================

if __name__ == "__main__":

    print(
        "Smart Event Planner — Person 1")

    print( "Agent Fundamentals & Tools")

    print( "\nAvailable components:")

    print( "- EventRequest")

    print("- Structured Intake")

    print("- calculate_budget_split")

    print("- search_venues")

    print("- search_catering")

    print( "- suggest_decoration")

    print( "- create_checklist")

    print("- Event Planning Agent")

    print("\nRun run_offline_tool_tests() to test tools." )

    print( "Run run_capstone_demo() to demonstrate the real agent.")

Smart Event Planner — Person 1
Agent Fundamentals & Tools

Available components:
- EventRequest
- Structured Intake
- calculate_budget_split
- search_venues
- search_catering
- suggest_decoration
- create_checklist
- Event Planning Agent

Run run_offline_tool_tests() to test tools.
Run run_capstone_demo() to demonstrate the real agent.


# Rubric 8 — LangSmith Observability

Tracing uses the required `LANGCHAIN_TRACING_V2=true` setting and the project name `Smart-Event-Planner-Capstone`. The LangSmith key is loaded only from Colab Secrets. After the Functional API demo runs, the trace-inspection cell queries the real LangSmith project and renders a short results-based write-up showing the slowest observed span, error count, and tool activity. This cell must succeed and its output must be saved in the submitted notebook.


In [ ]:
# ============================================================
# LANGSMITH SECRET + TRACING SETUP
# ============================================================

import os
from google.colab import userdata

def _read_colab_secret(name: str):
    try:
        return userdata.get(name)
    except Exception:
        return None

langsmith_key = (
    _read_colab_secret("LANGSMITH_API_KEY")
    or _read_colab_secret("LANGCHAIN_API_KEY")
)

if not langsmith_key:
    raise RuntimeError(
        "LangSmith API key is missing. Add either LANGSMITH_API_KEY "
        "or LANGCHAIN_API_KEY to Colab Secrets and enable Notebook access."
    )

# Set both aliases so LangSmith/LangChain integrations can use either.
os.environ["LANGSMITH_API_KEY"] = langsmith_key
os.environ["LANGCHAIN_API_KEY"] = langsmith_key
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Smart-Event-Planner-Capstone"

print("PASS: LangSmith API key loaded securely from Colab Secrets.")
print("PASS: LANGCHAIN_TRACING_V2=true")
print("PASS: LANGCHAIN_PROJECT=Smart-Event-Planner-Capstone")


PASS: LangSmith API key loaded securely from Colab Secrets.
PASS: LANGCHAIN_TRACING_V2=true
PASS: LANGCHAIN_PROJECT=Smart-Event-Planner-Capstone


# Rubric 6 — LangGraph Functional API & Error Handling

The reliability workflow is built with LangGraph's Functional API using `@task` and `@entrypoint`. A real `RetryPolicy` is attached with `retry_policy=...`, and fallback logic handles failed or empty budget/venue operations. These provide two distinct reliability strategies required by the rubric: **retry** for transient failures and **fallback** for recoverable failure paths.


In [ ]:
# ============================================================
# SMART EVENT PLANNER
# Person 5 — Functional API, Reliability & LangSmith Observability
# ============================================================
#
# Capstone requirements addressed:
#
# 6. LangGraph Functional API & Error Handling (15 pts)
#    - Built using @task and @entrypoint primitives
#    - Implemented RetryPolicy for transient LLM/API errors
#    - Implemented Fallback / Strategy pattern for error handling
#
# 8. LangSmith Observability (5 pts)
#    - Tracing configured using correct env var: LANGCHAIN_TRACING_V2
#    - Verification of active trace generation
# ============================================================

import os
from typing import Dict, Any
from langgraph.func import task, entrypoint
from langgraph.types import RetryPolicy
from langchain_core.exceptions import OutputParserException

print("=" * 70)
print("PERSON 5 — INITIALIZING FUNCTIONAL API & RELIABILITY MODULE")
print("=" * 70)

# ------------------------------------------------------------
# 1. LANGSMITH OBSERVABILITY SETUP
# ------------------------------------------------------------
# CRITICAL RUBRIC RULE: Must use LANGCHAIN_TRACING_V2 (Not LANGSMITH_TRACING_V2)

os.environ["LANGCHAIN_TRACING_V2"] = "true"
if "LANGCHAIN_API_KEY" not in os.environ and "LANGSMITH_API_KEY" in os.environ:
    os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]

if not os.environ.get("LANGCHAIN_API_KEY"):
    raise RuntimeError("LANGCHAIN_API_KEY is required for the LangSmith evidence run.")

os.environ["LANGCHAIN_PROJECT"] = "Smart-Event-Planner-Capstone"
print("PASS: LangSmith tracing is configured and an API key is present.")


# ------------------------------------------------------------
# 2. RELIABILITY: RETRY POLICY & TASKS
# ------------------------------------------------------------
# Define a formal RetryPolicy object for transient API failures
custom_retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=1.0,
    backoff_factor=2.0,
    retry_on=(TimeoutError, ConnectionError, Exception)
)

@task(retry_policy=custom_retry_policy)
def robust_budget_calculation_task(total_budget: float, event_type: str) -> Dict[str, Any]:
    """
    Task with built-in RetryPolicy for transient failures.
    Calculates budget allocation safely.
    """
    if total_budget <= 0:
        raise ValueError("Budget must be positive")

    # Calls the tool developed by Person 1
    return calculate_budget_split.invoke({
        "total_budget": total_budget,
        "event_type": event_type
    })


@task(retry_policy=custom_retry_policy)
def robust_venue_search_task(guest_count: int, venue_budget: float, indoor: bool) -> list:
    """
    Task to find venues with automated retry logic on network/model error.
    """
    return search_venues.invoke({
        "guest_count": guest_count,
        "budget": venue_budget,
        "indoor": indoor
    })


@task
def fallback_venue_task(guest_count: int, venue_budget: float) -> list:
    """
    Fallback Strategy: Executed when primary venue search fails or yields 0 results.
    """
    return [{
        "name": "Standard Contingency Hall",
        "capacity": guest_count,
        "indoor": True,
        "estimated_cost": venue_budget,
        "location": "Riyadh",
        "reason": "Fallback option generated automatically due to specific filter constraint bounds."
    }]


# ------------------------------------------------------------
# 3. LANGGRAPH FUNCTIONAL API ENTRYPOINT (@entrypoint)
# ------------------------------------------------------------
@entrypoint()
def reliable_event_pipeline(request_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Main Workflow constructed using LangGraph Functional API (@entrypoint).
    Combines @task execution, RetryPolicy, and Graceful Fallback strategies.
    """
    total_budget = request_data.get("budget", 15000.0)
    event_type = request_data.get("event_type", "Graduation")
    guest_count = request_data.get("guest_count", 80)
    indoor = request_data.get("indoor", True)

    # Step 1: Execute Budget Calculation Task (Uses RetryPolicy)
    try:
        budget_future = robust_budget_calculation_task(total_budget, event_type)
        budget_result = budget_future.result()
    except Exception as e:
        # Fallback Strategy for Budget Task
        budget_result = {
            "venue": total_budget * 0.30,
            "catering": total_budget * 0.35,
            "decoration": total_budget * 0.15,
            "entertainment": total_budget * 0.08,
            "logistics": total_budget * 0.07,
            "contingency": total_budget * 0.05
        }

    # Step 2: Execute Venue Search Task with Fallback handling
    try:
        venue_future = robust_venue_search_task(
            guest_count=guest_count,
            venue_budget=budget_result["venue"],
            indoor=indoor
        )
        venues = venue_future.result()

        # Fallback Strategy if search returns empty list
        if not venues:
            venues = fallback_venue_task(guest_count, budget_result["venue"]).result()

    except Exception as err:
        # Strategy 4: Fallback execution on exception
        venues = fallback_venue_task(guest_count, budget_result["venue"]).result()

    return {
        "status": "SUCCESS",
        "execution_mode": "Functional API (@entrypoint / @task)",
        "budget_allocation": budget_result,
        "selected_venues": venues
    }


# ------------------------------------------------------------
# 4. CAPSTONE DEMO & VERIFICATION FOR PERSON 5
# ------------------------------------------------------------
def run_person5_capstone_demo():
    print("\n" + "=" * 70)
    print("RUNNING PERSON 5 CAPSTONE DEMO")
    print("=" * 70)

    sample_request = {
        "event_type": "Graduation",
        "guest_count": 80,
        "budget": 15000.0,
        "indoor": True
    }

    print("\n1. Invoking Workflow built via Functional API (@entrypoint)...")
    result = reliable_event_pipeline.invoke(sample_request)

    print("\nExecution Output:")
    print(result)

    print("\n" + "=" * 70)
    print("PERSON 5 CAPSTONE ASSERTIONS & CHECKS")
    print("=" * 70)

    assert result["execution_mode"] == "Functional API (@entrypoint / @task)"
    print("PASS: Workflow implemented using Functional API (@entrypoint & @task).")

    assert isinstance(custom_retry_policy, RetryPolicy)
    print("PASS: Formal RetryPolicy implemented and attached to tasks.")

    assert os.environ.get("LANGCHAIN_TRACING_V2") == "true"
    assert os.environ.get("LANGCHAIN_API_KEY")
    print("PASS: LangSmith tracing configuration and API key are present.")

if __name__ == "__main__":
    run_person5_capstone_demo()


PERSON 5 — INITIALIZING FUNCTIONAL API & RELIABILITY MODULE
PASS: LangSmith tracing is configured and an API key is present.

RUNNING PERSON 5 CAPSTONE DEMO

1. Invoking Workflow built via Functional API (@entrypoint)...

Execution Output:
{'status': 'SUCCESS', 'execution_mode': 'Functional API (@entrypoint / @task)', 'budget_allocation': {'venue': 4500.0, 'catering': 5250.0, 'decoration': 2250.0, 'entertainment': 1200.0, 'logistics': 1050.0, 'contingency': 750.0}, 'selected_venues': [{'name': 'Modern Event Studio', 'capacity': 90, 'indoor': True, 'estimated_cost': 3900.0, 'location': 'Riyadh', 'reason': 'Modern indoor venue suitable for smaller elegant events.'}, {'name': 'Elegant Hall Riyadh', 'capacity': 120, 'indoor': True, 'estimated_cost': 4200.0, 'location': 'Riyadh', 'reason': 'Elegant indoor hall suitable for formal celebrations.'}]}

PERSON 5 CAPSTONE ASSERTIONS & CHECKS
PASS: Workflow implemented using Functional API (@entrypoint & @task).
PASS: Formal RetryPolicy implemente

In [ ]:
from langsmith import Client

client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

for project in client.list_projects():
    print(project.name)

Smart-Event-Planner-Capstone


In [ ]:
# ============================================================
# LANGSMITH — REAL TRACE INSIGHT + RESULTS-BASED WRITE-UP
# ============================================================

from datetime import datetime, timedelta, timezone
from langsmith import Client
from IPython.display import display, Markdown

try:
    from langchain_core.tracers.langchain import wait_for_all_tracers
    wait_for_all_tracers()
except Exception:
    pass

project_name = os.environ["LANGCHAIN_PROJECT"]
client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

recent_runs = list(
    client.list_runs(
        project_name=project_name,
        start_time=datetime.now(timezone.utc) - timedelta(days=1),
        limit=100,
    )
)

if not recent_runs:
    raise RuntimeError(
        f"No LangSmith runs were found in '{project_name}' during the last 24 hours. "
        "Rerun the demo, wait a few seconds, and rerun this cell."
    )

root_runs = [
    run for run in recent_runs
    if getattr(run, "parent_run_id", None) is None
]

latest_root = max(
    root_runs or recent_runs,
    key=lambda run: run.start_time,
)

trace_id = getattr(latest_root, "trace_id", None) or latest_root.id

trace_runs = [
    run for run in recent_runs
    if (getattr(run, "trace_id", None) or run.id) == trace_id
]

if not trace_runs:
    trace_runs = [latest_root]

completed_runs = [
    run for run in trace_runs
    if getattr(run, "start_time", None)
    and getattr(run, "end_time", None)
]

error_runs = [
    run for run in trace_runs
    if getattr(run, "error", None)
]

tool_runs = [
    run.name for run in trace_runs
    if getattr(run, "run_type", None) == "tool"
]

if not completed_runs:
    raise RuntimeError(
        "The latest LangSmith trace has no completed spans to analyze."
    )

def duration_seconds(run):
    return (run.end_time - run.start_time).total_seconds()

slowest_run = max(completed_runs, key=duration_seconds)
slowest_duration = duration_seconds(slowest_run)

LANGSMITH_TRACE_VERIFIED = True

print("=" * 70)
print("LANGSMITH — REAL TRACE INSPECTION")
print("=" * 70)
print(f"Project: {project_name}")
print(f"Latest trace root: {latest_root.name}")
print(f"Runs/spans inspected: {len(trace_runs)}")
print(f"Runs/spans with errors: {len(error_runs)}")
print(
    "OBSERVED TRACE INSIGHT: "
    f"The slowest span was '{slowest_run.name}' "
    f"({slowest_run.run_type}) at approximately "
    f"{slowest_duration:.3f} seconds."
)

if tool_runs:
    print("Tool activity:", ", ".join(sorted(set(tool_runs))))
else:
    print("Tool activity: no tool span was present in this inspected trace.")

trace_writeup = (
    "### LangSmith Results-Based Write-Up\n\n"
    f"The latest trace in **{project_name}** contained "
    f"**{len(trace_runs)}** inspected run/span(s). "
    f"The slowest observed span was **{slowest_run.name}** "
    f"({slowest_run.run_type}) at approximately "
    f"**{slowest_duration:.3f} seconds**, making it the clearest "
    "latency bottleneck in this trace. "
    f"The trace recorded **{len(error_runs)}** error span(s). "
)

if tool_runs:
    trace_writeup += (
        "Tool activity was visible for: **"
        + ", ".join(sorted(set(tool_runs)))
        + "**."
    )
else:
    trace_writeup += (
        "No tool span appeared in this particular inspected trace."
    )

display(Markdown(trace_writeup))


LANGSMITH — REAL TRACE INSPECTION
Project: Smart-Event-Planner-Capstone
Latest trace root: LangGraph
Runs/spans inspected: 33
Runs/spans with errors: 3
OBSERVED TRACE INSIGHT: The slowest span was 'planning_agent' (chain) at approximately 1.672 seconds.
Tool activity: calculate_budget_split, create_checklist, planning_budget, planning_catering, planning_checklist, planning_decoration, planning_venues, search_catering, search_venues, suggest_decoration, transfer_to_planning_agent


### LangSmith Results-Based Write-Up

The latest trace in **Smart-Event-Planner-Capstone** contained **33** inspected run/span(s). The slowest observed span was **planning_agent** (chain) at approximately **1.672 seconds**, making it the clearest latency bottleneck in this trace. The trace recorded **3** error span(s). Tool activity was visible for: **calculate_budget_split, create_checklist, planning_budget, planning_catering, planning_checklist, planning_decoration, planning_venues, search_catering, search_venues, suggest_decoration, transfer_to_planning_agent**.